# 04a · 장애물 분포 재분할과 v4 shard

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** 기존 test를 고정한 채 train/validation 그룹을 교환해 장애물 다중 레이블 분포 오차를 줄입니다.
- **입력:** 기존 v3 split/Manifest, full shard와 익명 test seal
- **출력:** v4 split seal, 장애물 분포 감사표와 train/validation dataset seal
- **실제 저장된 실행 상태:** 실자료 분할 최적화의 저장 출력은 objective=20.466860227779286→8.074057890211245, accepted swaps=2,736이며, MULTILABEL SPLIT FROZEN과 V4 DATASET COMPLETE를 기록합니다. 앞의 synthetic_self_test() 출력인 objective=26.21866020761246→21.374194232987314, accepted_swaps=9는 합성 검증 결과이며 실자료 최적화 결과와 구분합니다. train/val shard 수는 177/175입니다. 기존 test를 고정한 후속 실험이며 새로운 외부 test를 만든 기록은 아닙니다. 원래 seal 파일은 공개 저장소에 없어 저장 로그와 독립적으로 대조하지 못했습니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/04a_obstacle_rebalance.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/04a_obstacle_rebalance.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 Mission 3 — 04A. 장애물 다중 레이블 재분할과 v4 학습 shard

이 노트북은 기존 v3 test 30,809장을 고정하고 train·validation 그룹만 재배정합니다.
`type`, 15개 장애물 multi-hot, `type×obstacle`, `type×water-decile`, 장애물 개수와
품질 플래그의 분포 오차를 줄이되 SHA group, 전체 수량, 유형별 수량을 유지합니다.

실행 결과는 새 `v4_ml_fixedtest_v2` split seal과 `m3ds-v4-ml-fixedtest-v2` train/validation
WebDataset seal입니다. 기존 v3 및 `m3ds-v2-pilot` 객체는 읽기만 수행합니다.

권장 런타임은 **CPU 고용량 RAM**입니다. shard 재포장은 원본 ZIP을 다시 풀지 않고 기존에
검증된 full train/validation tar의 image byte를 그대로 복사합니다. 중단 후 같은 설정으로
재실행하면 input shard별 commit을 검증하고 이어갑니다.


In [ ]:
%pip -q install "google-cloud-storage>=2.19,<4" "pyarrow==23.0.1" "pandas==2.3.3" "numpy==2.1.3"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 116.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.


In [ ]:
import gc
import hashlib
import io
import json
import math
import os
import shutil
import tarfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from google.colab import auth
from IPython.display import display

PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BUCKET_NAME = "YOUR_GCS_BUCKET"
INPUT_VERSION = "v2"
BASE_SPLIT_VERSION = "v3"
OUTPUT_VERSION = "v4_ml_fixedtest_v2"
BASE_DATASET_VERSION = "m3ds-v2-pilot"
DATASET_VERSION = "m3ds-v4-ml-fixedtest-v2"
RUN_ID = "m3-v4-multilabel-split-fixedtest-v2"
SERIALIZATION_VERSIONS = {
    "numpy": "2.1.3",
    "pandas": "2.3.3",
    "pyarrow": "23.0.1",
}
installed_serialization_versions = {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pyarrow": pa.__version__,
}
if installed_serialization_versions != SERIALIZATION_VERSIONS:
    raise RuntimeError(
        "직렬화 패키지 버전 불일치: "
        f"expected={SERIALIZATION_VERSIONS}, actual={installed_serialization_versions}"
    )
SEED = 20260902
SWAP_ITERATIONS = 300_000
SWAP_RESTARTS = 3
MIN_PAIR_GROUP_SUPPORT = 100

TYPE_ORDER = list("ABCEFGHS")
REAL_OBSTRUCTIONS = [
    "거센물살", "거울반사", "기타", "달빛반사", "반투명",
    "부유물", "부착물", "사람", "수풀", "안개", "탁류",
    "표면손상", "표면오염", "야간", "와류",
]
EXPECTED_RAW_ROWS = 308_085
EXPECTED_COUNTS = {"train": 246_468, "val": 30_808, "test": 30_809}
EXPECTED_EFFECTIVE_COUNTS = {"train": 246_308, "val": 30_808}

LOCAL_ROOT = Path("/content/dcc_mission3_04a")
LOCAL_INPUT = LOCAL_ROOT / "input"
LOCAL_OUTPUT = LOCAL_ROOT / "output"
LOCAL_STAGE = LOCAL_ROOT / "stage"
LOCAL_META = LOCAL_ROOT / "meta"
for path in [LOCAL_INPUT, LOCAL_OUTPUT, LOCAL_STAGE, LOCAL_META]:
    path.mkdir(parents=True, exist_ok=True)

MANIFEST_SEAL_OBJECT = f"dcc/audits/manifest_seal_{INPUT_VERSION}.json"
BASE_SPLIT_SEAL_OBJECT = (
    f"dcc/eval_private/manifests/split_seal_{BASE_SPLIT_VERSION}.json"
)
BASE_DATASET_SEAL_OBJECT = (
    f"dcc/mission3/datasets/{BASE_DATASET_VERSION}/dataset_seal.json"
)
BASE_INFERENCE_SEAL_OBJECT = (
    f"dcc/inference/mission3/{BASE_DATASET_VERSION}/test_image_seal.json"
)
SPLIT_PREFIX = f"dcc/eval_private/mission3/v4/{OUTPUT_VERSION}"
DATASET_PREFIX = f"dcc/mission3/datasets/{DATASET_VERSION}"
print({
    "output_version": OUTPUT_VERSION,
    "dataset_version": DATASET_VERSION,
    "fixed_test": EXPECTED_COUNTS["test"],
    "swap_iterations": SWAP_ITERATIONS,
})


{'output_version': 'v4_ml_fixedtest_v2', 'dataset_version': 'm3ds-v4-ml-fixedtest-v2', 'fixed_test': 30809, 'swap_iterations': 300000}


In [ ]:
from google.api_core.exceptions import NotFound, PreconditionFailed
from google.cloud import storage

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def canonical_sha256(value):
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def canonical_target_sha256(frame, target_column):
    """원본 float64와 DataLoader float32가 같은 수위를 같은 byte 계약으로 해시합니다."""
    required = {"stem", "type", target_column}
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"target hash 필수 열 누락: {sorted(missing)}")
    ordered = frame.sort_values("stem", kind="mergesort")
    records = [
        {
            "stem": str(getattr(row, "stem")),
            "type": str(getattr(row, "type")),
            "target_m_float32": float(np.float32(float(getattr(row, target_column)))),
        }
        for row in ordered.itertuples(index=False)
    ]
    return canonical_sha256(records)

def parse_gs_uri(uri):
    if not str(uri).startswith("gs://"):
        raise ValueError(f"GCS URI가 아닙니다: {uri}")
    bucket_name, object_name = str(uri)[5:].split("/", 1)
    return bucket_name, object_name

def download_object(bucket, object_name, local_path, generation=None):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {"checksum": "auto"}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    blob.download_to_filename(str(local_path), **kwargs)
    return local_path

def upload_create_only(bucket, local_path, object_name, artifact_type):
    local_path = Path(local_path)
    file_sha = sha256_file(local_path)
    blob = bucket.blob(object_name)
    blob.metadata = {
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }
    try:
        blob.upload_from_filename(
            str(local_path), if_generation_match=0, checksum="auto"
        )
    except PreconditionFailed:
        blob.reload()
        remote_sha = (blob.metadata or {}).get("sha256")
        remote_artifact_type = (blob.metadata or {}).get("artifact_type")
        if (
            remote_sha != file_sha
            or remote_artifact_type != artifact_type
            or int(blob.size or -1) != local_path.stat().st_size
        ):
            raise RuntimeError(
                f"create-only 충돌: gs://{bucket.name}/{object_name}"
            )
    blob.reload()
    if (
        (blob.metadata or {}).get("sha256") != file_sha
        or (blob.metadata or {}).get("artifact_type") != artifact_type
    ):
        raise RuntimeError(f"업로드 SHA metadata 검증 실패: {object_name}")
    return {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }

def upload_json_create_only(bucket, payload, object_name, artifact_type):
    local_path = LOCAL_META / Path(object_name).name
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.write_text(
        json.dumps(
            payload, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False
        ) + "\n",
        encoding="utf-8",
    )
    return upload_create_only(bucket, local_path, object_name, artifact_type)

def download_json(bucket, object_name, generation=None):
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    return json.loads(blob.download_as_bytes(**kwargs).decode("utf-8"))

def remote_descriptor(bucket, object_name, artifact_type=None):
    blob = bucket.blob(object_name)
    blob.reload()
    metadata = blob.metadata or {}
    file_sha = metadata.get("sha256")
    if not file_sha:
        raise RuntimeError(f"remote object에 sha256 metadata가 없습니다: {object_name}")
    remote_artifact_type = metadata.get("artifact_type")
    if artifact_type is not None and remote_artifact_type != artifact_type:
        raise RuntimeError(
            f"remote artifact_type 불일치: {object_name} "
            f"({remote_artifact_type!r} != {artifact_type!r})"
        )
    return {
        "uri": f"gs://{bucket.name}/{blob.name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": remote_artifact_type or "unknown",
    }


## 1. 상위 seal과 byte 고정 입력 검증

raw manifest의 장애물 list와 v3 split을 `stem`으로 one-to-one 결합합니다. test stem 집합,
parent generation, byte 수와 SHA-256을 모두 고정합니다.


In [ ]:
def descriptor_size(descriptor):
    return int(descriptor.get("bytes", descriptor.get("size", -1)))

def download_verified_descriptor(descriptor, local_path, expected_prefix=None):
    bucket_name, object_name = parse_gs_uri(descriptor["uri"])
    if bucket_name != BUCKET_NAME:
        raise AssertionError("descriptor bucket 불일치")
    if expected_prefix is not None and not object_name.startswith(expected_prefix):
        raise AssertionError(f"descriptor prefix 불일치: {object_name}")
    download_object(
        BUCKET, object_name, local_path, generation=int(descriptor["generation"])
    )
    if Path(local_path).stat().st_size != descriptor_size(descriptor):
        raise AssertionError("descriptor byte 불일치")
    if sha256_file(local_path) != descriptor["sha256"]:
        raise AssertionError("descriptor SHA 불일치")
    remote = BUCKET.blob(object_name, generation=int(descriptor["generation"]))
    remote.reload()
    if int(remote.size) != descriptor_size(descriptor):
        raise AssertionError("pinned remote byte 불일치")
    metadata_sha = (remote.metadata or {}).get("sha256")
    if metadata_sha and metadata_sha != descriptor["sha256"]:
        raise AssertionError("remote SHA metadata 불일치")
    return Path(local_path)

auth.authenticate_user()
STORAGE_CLIENT = storage.Client(project=PROJECT_ID)
BUCKET = STORAGE_CLIENT.bucket(BUCKET_NAME)
BUCKET.reload()

manifest_seal = download_json(BUCKET, MANIFEST_SEAL_OBJECT)
base_split_seal = download_json(BUCKET, BASE_SPLIT_SEAL_OBJECT)
base_dataset_seal = download_json(BUCKET, BASE_DATASET_SEAL_OBJECT)
base_inference_seal = download_json(BUCKET, BASE_INFERENCE_SEAL_OBJECT)
for name, payload in {
    "manifest": manifest_seal,
    "split": base_split_seal,
    "dataset": base_dataset_seal,
    "inference": base_inference_seal,
}.items():
    assertions = payload.get("assertions", {})
    if assertions and not all(assertions.values()):
        raise AssertionError(f"{name} upstream assertion 실패")
if base_dataset_seal.get("status") != "DATASET_COMPLETE":
    raise AssertionError("base dataset seal 상태 오류")
if base_dataset_seal.get("data_profile") != "full":
    raise AssertionError("base dataset은 full profile이어야 합니다")
if base_inference_seal.get("status") != "INFERENCE_IMAGES_COMPLETE":
    raise AssertionError("base inference image seal 상태 오류")
if int(base_inference_seal["counts"]["test"]) != EXPECTED_COUNTS["test"]:
    raise AssertionError("base anonymous test count 불일치")

raw_descriptor = manifest_seal["artifacts"]["raw_manifest"]
split_descriptor = base_split_seal["artifacts"]["splits"]
RAW_PATH = download_verified_descriptor(
    raw_descriptor,
    LOCAL_INPUT / "raw_manifest_v2.parquet",
    "dcc/manifests/",
)
BASE_SPLIT_PATH = download_verified_descriptor(
    split_descriptor,
    LOCAL_INPUT / "splits_v3.parquet",
    "dcc/eval_private/manifests/",
)
MANIFEST_SEAL_DESCRIPTOR = remote_descriptor(
    BUCKET, MANIFEST_SEAL_OBJECT
)
BASE_SPLIT_SEAL_DESCRIPTOR = remote_descriptor(
    BUCKET, BASE_SPLIT_SEAL_OBJECT
)
BASE_DATASET_SEAL_DESCRIPTOR = remote_descriptor(
    BUCKET, BASE_DATASET_SEAL_OBJECT
)
BASE_INFERENCE_SEAL_DESCRIPTOR = remote_descriptor(
    BUCKET, BASE_INFERENCE_SEAL_OBJECT
)
BASE_TEST_KEY_MAP_DESCRIPTOR = base_dataset_seal["private"]["test_key_map"]
if "/eval_private/" not in BASE_TEST_KEY_MAP_DESCRIPTOR["uri"]:
    raise AssertionError("base test key map은 private prefix에 있어야 합니다")
BASE_TEST_KEY_MAP_PATH = download_verified_descriptor(
    BASE_TEST_KEY_MAP_DESCRIPTOR,
    LOCAL_INPUT / "base_test_key_map.parquet",
    "dcc/eval_private/mission3/",
)
print("verified raw:", raw_descriptor["uri"])
print("verified base split:", split_descriptor["uri"])


verified raw: gs://YOUR_GCS_BUCKET/dcc/manifests/raw_manifest_v2.parquet
verified base split: gs://YOUR_GCS_BUCKET/dcc/eval_private/manifests/splits_v3.parquet


## 2. 장애물 정규화와 그룹 보존 swap 최적화

`해당없음`, 빈 값, 모순 표본을 별도 플래그로 보존합니다. v3 train/validation에서 같은
group size와 같은 유형 count vector를 가진 그룹끼리만 교환하므로 test, 전체 수량,
유형별 수량과 quarantine 정책은 변하지 않습니다.


In [ ]:
"""Pure split utilities for the DCC Mission 3 v4 multilabel experiment.

The module has no GCS or Colab dependency.  The generated Colab notebook embeds
this source verbatim and the local QA script executes its synthetic self-test.
"""

from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from itertools import combinations
import re
from typing import Iterable, Sequence

import numpy as np
import pandas as pd


TYPE_ORDER = tuple("ABCEFGHS")
REAL_OBSTRUCTIONS = (
    "거센물살",
    "거울반사",
    "기타",
    "달빛반사",
    "반투명",
    "부유물",
    "부착물",
    "사람",
    "수풀",
    "안개",
    "탁류",
    "표면손상",
    "표면오염",
    "야간",
    "와류",
)


@dataclass(frozen=True)
class SplitOptimizationResult:
    group_ids: np.ndarray
    group_assignment: np.ndarray
    feature_names: tuple[str, ...]
    initial_objective: float
    final_objective: float
    accepted_swaps: int
    attempted_swaps: int
    restart: int

    def assignment_series(self) -> pd.Series:
        return pd.Series(
            self.group_assignment,
            index=pd.Index(self.group_ids, name="group_id"),
            name="split",
        )


def normalize_obstructions(
    value,
    vocabulary: Sequence[str] = REAL_OBSTRUCTIONS,
) -> dict:
    """Normalize one raw list while preserving ambiguity as explicit flags."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        raw_tags: list[str] = []
    elif isinstance(value, np.ndarray):
        raw_tags = value.tolist()
    elif isinstance(value, (list, tuple, set)):
        raw_tags = list(value)
    else:
        raise TypeError(f"obstruction must be list-like, got {type(value).__name__}")

    if any(not isinstance(item, str) for item in raw_tags):
        raise TypeError("obstruction contains a non-string item")
    stripped = [item.strip() for item in raw_tags]
    allowed = set(vocabulary)
    observed = set(stripped)
    # Preserve the sealed vocabulary order instead of locale-dependent sorting.
    real = [item for item in vocabulary if item in observed]
    has_none = "해당없음" in stripped
    has_empty = any(item == "" for item in stripped) or not stripped
    unknown = sorted(
        {item for item in stripped if item and item != "해당없음" and item not in allowed}
    )
    contradiction = bool(has_none and real)
    unknown_flag = bool(has_empty or unknown)
    return {
        "obstruction_labels": real,
        # Quality flags are independent: a row may be both none-only and carry
        # an empty/unknown token that must remain visible to audit code.
        "none_only": bool(has_none and not real),
        "multi_obstacle": len(real) >= 2,
        "contradiction": contradiction,
        "empty_or_unknown": unknown_flag,
        "unknown_obstruction_labels": unknown,
        "obstruction_targets": [int(name in real) for name in vocabulary],
        # Contradictory or unknown labels may still support regression, but they do
        # not contribute to the auxiliary multilabel loss.
        "obstruction_mask": [int(not contradiction and not unknown_flag)] * len(vocabulary),
    }


def add_normalized_obstruction_columns(
    frame: pd.DataFrame,
    source_column: str = "obstruction",
    vocabulary: Sequence[str] = REAL_OBSTRUCTIONS,
) -> pd.DataFrame:
    if source_column not in frame:
        raise KeyError(source_column)
    normalized = [normalize_obstructions(value, vocabulary) for value in frame[source_column]]
    out = frame.copy()
    for column in normalized[0] if normalized else ():
        out[column] = [item[column] for item in normalized]
    return out


def _water_bin_as_int(value) -> int:
    if pd.isna(value):
        raise ValueError("water_bin is missing")
    if isinstance(value, (bool, np.bool_)):
        raise ValueError(f"boolean is not a water bin: {value!r}")
    if isinstance(value, str):
        match = re.fullmatch(r"\s*(?:D)?0?([0-9])\s*", value)
        if match is None:
            raise ValueError(f"cannot parse water bin: {value!r}")
        parsed = int(match.group(1))
    elif isinstance(value, (int, np.integer)):
        parsed = int(value)
    elif isinstance(value, (float, np.floating)):
        if not np.isfinite(value) or not float(value).is_integer():
            raise ValueError(f"water bin must be an integer: {value!r}")
        parsed = int(value)
    else:
        raise TypeError(f"unsupported water bin type: {type(value).__name__}")
    if not 0 <= parsed <= 9:
        raise ValueError(f"water bin outside 0..9: {parsed}")
    return parsed


def build_row_feature_matrix(
    frame: pd.DataFrame,
    type_order: Sequence[str] = TYPE_ORDER,
    vocabulary: Sequence[str] = REAL_OBSTRUCTIONS,
    *,
    min_pair_group_support: int = 100,
) -> tuple[np.ndarray, tuple[str, ...], tuple[str, ...]]:
    """Build type, obstacle, interaction, water-bin, and quality features."""
    required = {
        "type",
        "water_bin",
        "obstruction_labels",
        "none_only",
        "multi_obstacle",
        "contradiction",
        "empty_or_unknown",
    }
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"feature columns missing: {sorted(missing)}")
    if min_pair_group_support < 1:
        raise ValueError("min_pair_group_support must be positive")
    if "group_id" not in frame.columns:
        raise AssertionError("group_id is required for obstacle-pair support")

    vocabulary = tuple(vocabulary)
    vocabulary_set = set(vocabulary)
    normalized_label_rows: list[tuple[str, ...]] = []
    for raw_labels in frame["obstruction_labels"]:
        observed = set(raw_labels)
        unknown = observed - vocabulary_set
        if unknown:
            raise AssertionError(
                f"unknown normalized obstruction: {sorted(unknown)}"
            )
        normalized_label_rows.append(
            tuple(name for name in vocabulary if name in observed)
        )

    group_codes, _ = pd.factorize(frame["group_id"].astype(str), sort=True)
    pair_group_support: dict[tuple[str, str], set[int]] = {}
    for labels, group_code in zip(normalized_label_rows, group_codes):
        for pair in combinations(labels, 2):
            pair_group_support.setdefault(pair, set()).add(int(group_code))
    frequent_pairs = tuple(
        pair
        for pair in combinations(vocabulary, 2)
        if len(pair_group_support.get(pair, ())) >= int(min_pair_group_support)
    )

    feature_names: list[str] = []
    families: list[str] = []

    def add(name: str, family: str) -> int:
        feature_names.append(name)
        families.append(family)
        return len(feature_names) - 1

    type_col = {name: add(f"type={name}", "type") for name in type_order}
    obs_col = {name: add(f"obs={name}", "obstacle") for name in vocabulary}
    pair_col = {
        pair: add(f"obs_pair={pair[0]}|{pair[1]}", "obstacle_pair")
        for pair in frequent_pairs
    }
    type_obs_col = {
        (type_name, obs): add(f"type_obs={type_name}|{obs}", "type_obstacle")
        for type_name in type_order
        for obs in vocabulary
    }
    type_water_col = {
        (type_name, water_bin): add(
            f"type_water={type_name}|D{water_bin}", "type_water"
        )
        for type_name in type_order
        for water_bin in range(10)
    }
    flag_names = ("none_only", "multi_obstacle", "contradiction", "empty_or_unknown")
    flag_col = {name: add(f"flag={name}", "quality_flag") for name in flag_names}
    cardinality_col = {
        label: add(f"obstacle_count={label}", "obstacle_count")
        for label in ("0", "1", "2", "3+")
    }

    matrix = np.zeros((len(frame), len(feature_names)), dtype=np.uint8)
    type_to_index = {name: index for index, name in enumerate(type_order)}
    for row_index, row in enumerate(frame.itertuples(index=False)):
        type_name = str(getattr(row, "type"))
        if type_name not in type_to_index:
            raise AssertionError(f"unknown type: {type_name}")
        matrix[row_index, type_col[type_name]] = 1
        labels = normalized_label_rows[row_index]
        for label in labels:
            matrix[row_index, obs_col[label]] = 1
            matrix[row_index, type_obs_col[(type_name, label)]] = 1
        for pair in combinations(labels, 2):
            column = pair_col.get(pair)
            if column is not None:
                matrix[row_index, column] = 1
        water_bin = _water_bin_as_int(getattr(row, "water_bin"))
        matrix[row_index, type_water_col[(type_name, water_bin)]] = 1
        for flag in flag_names:
            if bool(getattr(row, flag)):
                matrix[row_index, flag_col[flag]] = 1
        count = len(labels)
        count_label = "3+" if count >= 3 else str(count)
        matrix[row_index, cardinality_col[count_label]] = 1

    return matrix, tuple(feature_names), tuple(families)



def _sum_binary_features_by_group(
    group_codes: np.ndarray,
    row_features: np.ndarray,
    group_count: int,
    group_sizes: np.ndarray,
) -> np.ndarray:
    """Sum a dense binary row matrix without np.add.at's large cast temporary.

    The base v4 matrix has 231 columns plus selected obstacle pairs.
    Column-wise bincount performs the
    aggregation in compiled code and keeps peak memory predictable.  Unsigned
    width is selected from the largest group, so a future scene group cannot
    silently overflow int16.
    """
    if row_features.ndim != 2 or len(group_codes) != len(row_features):
        raise AssertionError("invalid row feature matrix shape")
    if row_features.size and (
        not np.issubdtype(row_features.dtype, np.number)
        or int(row_features.min()) < 0
        or int(row_features.max()) > 1
    ):
        raise AssertionError("row features must be a binary numeric matrix")
    max_group_size = int(group_sizes.max(initial=0))
    dtype = np.uint16 if max_group_size <= np.iinfo(np.uint16).max else np.uint32
    result = np.empty((group_count, row_features.shape[1]), dtype=dtype)
    for feature_index in range(row_features.shape[1]):
        counts = np.bincount(
            group_codes,
            weights=row_features[:, feature_index],
            minlength=group_count,
        )
        if counts.size != group_count or np.any(counts < 0):
            raise AssertionError("group feature aggregation failed")
        result[:, feature_index] = counts.astype(dtype, copy=False)
    return result

def aggregate_group_features(
    frame: pd.DataFrame,
    row_features: np.ndarray,
    *,
    eligibility_column: str = "eligible_for_training",
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    required = {"group_id", "split", eligibility_column}
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"group columns missing: {sorted(missing)}")
    if len(frame) != len(row_features):
        raise AssertionError("row feature length mismatch")

    group_codes, group_ids = pd.factorize(frame["group_id"].astype(str), sort=True)
    group_count = len(group_ids)
    group_sizes = np.bincount(group_codes, minlength=group_count).astype(np.int32)
    group_features = _sum_binary_features_by_group(
        group_codes, row_features, group_count, group_sizes
    )

    split_codes = pd.Categorical(
        frame["split"], categories=["train", "val", "test"]
    ).codes
    if (split_codes < 0).any():
        raise AssertionError("unexpected base split")
    group_split_min = np.full(group_count, 3, dtype=np.int8)
    group_split_max = np.full(group_count, -1, dtype=np.int8)
    np.minimum.at(group_split_min, group_codes, split_codes)
    np.maximum.at(group_split_max, group_codes, split_codes)
    if not np.array_equal(group_split_min, group_split_max):
        raise AssertionError("base split already leaks a group")
    split_names = np.asarray(["train", "val", "test"], dtype=object)
    group_splits = split_names[group_split_min]

    if frame[eligibility_column].isna().any():
        raise AssertionError(f"{eligibility_column} contains null")
    eligibility_values = set(frame[eligibility_column].unique().tolist())
    if not eligibility_values <= {True, False, 0, 1}:
        raise AssertionError(f"{eligibility_column} must be boolean")
    eligible = frame[eligibility_column].astype(bool).to_numpy()
    eligible_sum = np.bincount(
        group_codes, weights=eligible.astype(np.int16), minlength=group_count
    ).astype(np.int32)
    group_all_eligible = eligible_sum == group_sizes
    return (
        np.asarray(group_ids, dtype=object),
        group_codes,
        group_features,
        group_sizes,
        np.asarray(group_splits, dtype=object),
        group_all_eligible,
    )



def _group_all_true(
    frame: pd.DataFrame,
    column: str,
    group_codes: np.ndarray,
    group_sizes: np.ndarray,
) -> np.ndarray:
    """Return whether every row in each factorized group is true.

    Swap eligibility and holdout eligibility have different meanings.  This
    helper derives an additional group-level predicate without aggregating the
    relatively large feature matrix a second time.
    """
    if column not in frame.columns:
        raise AssertionError(f"missing group predicate: {column}")
    if len(group_codes) != len(frame):
        raise AssertionError("group predicate row length mismatch")
    if frame[column].isna().any():
        raise AssertionError(f"{column} contains null")
    values = set(frame[column].unique().tolist())
    if not values <= {True, False, 0, 1}:
        raise AssertionError(f"{column} must be boolean")
    row_true = frame[column].astype(bool).to_numpy()
    true_sum = np.bincount(
        group_codes,
        weights=row_true.astype(np.int8),
        minlength=len(group_sizes),
    ).astype(np.int32)
    if len(true_sum) != len(group_sizes):
        raise AssertionError("group predicate aggregation failed")
    return true_sum == group_sizes


def _objective_weights(
    target_val: np.ndarray,
    families: Sequence[str],
) -> np.ndarray:
    family_weight = {
        "type": 8.0,
        "obstacle": 3.0,
        "obstacle_pair": 0.75,
        "type_obstacle": 2.0,
        "type_water": 2.5,
        "quality_flag": 1.5,
        "obstacle_count": 1.5,
    }
    base = np.asarray([family_weight[name] for name in families], dtype=np.float64)
    # A five-row floor caps the influence of extremely rare cells.
    denominator = np.maximum(target_val.astype(np.float64), 5.0) ** 2
    return base / denominator


def _objective(error: np.ndarray, weights: np.ndarray) -> float:
    return float(np.dot(weights, error.astype(np.float64) ** 2))


def optimize_fixed_test_group_swaps(
    frame: pd.DataFrame,
    row_features: np.ndarray,
    feature_names: Sequence[str],
    families: Sequence[str],
    *,
    target_val_rows: int,
    iterations: int = 300_000,
    restarts: int = 3,
    seed: int = 20260902,
    progress_every: int | None = None,
    swap_eligible_column: str = "eligible_for_training",
    holdout_eligibility_column: str = "eligible_for_training",
) -> SplitOptimizationResult:
    """Improve multilabel balance while preserving the exact v3 test and counts.

    A proposal swaps one all-eligible train group and one validation group with
    the same row count and the same type-count vector.  Every accepted proposal
    therefore preserves total rows, per-type rows, group integrity, and the
    train-only quarantine policy exactly.  ``swap_eligible_column`` controls
    whether a group may move, while ``holdout_eligibility_column`` controls
    whether it is valid in validation.  A label-conflict group may therefore
    remain frozen in its inherited validation split without being confused
    with an actually ineligible training group.
    """
    if len(feature_names) != row_features.shape[1] or len(families) != row_features.shape[1]:
        raise AssertionError("feature metadata length mismatch")
    if iterations <= 0 or restarts <= 0:
        raise ValueError("iterations and restarts must be positive")

    (
        group_ids,
        _group_codes,
        group_features,
        group_sizes,
        base_splits,
        group_swap_eligible,
    ) = aggregate_group_features(
        frame,
        row_features,
        eligibility_column=swap_eligible_column,
    )
    group_holdout_eligible = _group_all_true(
        frame,
        holdout_eligibility_column,
        _group_codes,
        group_sizes,
    )
    if np.any(group_swap_eligible & ~group_holdout_eligible):
        raise AssertionError(
            "swap eligibility cannot override holdout/training ineligibility"
        )
    group_movable = group_swap_eligible & group_holdout_eligible

    type_columns = [index for index, name in enumerate(feature_names) if name.startswith("type=")]
    if len(type_columns) != len(TYPE_ORDER):
        raise AssertionError("type feature count mismatch")
    type_counts = group_features[:, type_columns]

    dev_mask = base_splits != "test"
    dev_rows = int(group_sizes[dev_mask].sum())
    if not 0 < target_val_rows < dev_rows:
        raise ValueError("target validation row count is outside development rows")
    # Optimize the same global 8:1:1 count objective reported by the audit.
    # Test is immutable, so its error is a constant.  A train<->val swap changes
    # both development errors in opposite directions and both terms must be
    # included in the delta.
    total_rows = int(group_sizes.sum())
    all_feature_total = group_features.sum(axis=0).astype(np.float64)
    base_test_mask = base_splits == "test"
    test_rows = int(group_sizes[base_test_mask].sum())
    target_train_rows = total_rows - int(target_val_rows) - test_rows
    target_train = all_feature_total * (target_train_rows / total_rows)
    target_val = all_feature_total * (target_val_rows / total_rows)
    target_test = all_feature_total * (test_rows / total_rows)
    weights = _objective_weights(target_val, families)

    base_train_mask = base_splits == "train"
    base_val_mask = base_splits == "val"
    if int(group_sizes[base_val_mask].sum()) != int(target_val_rows):
        raise AssertionError("base validation row count differs from target")
    if not group_holdout_eligible[base_val_mask].all():
        raise AssertionError("base validation contains an ineligible group")
    base_train_counts = group_features[base_train_mask].sum(axis=0).astype(np.float64)
    base_val_counts = group_features[base_val_mask].sum(axis=0).astype(np.float64)
    base_test_counts = group_features[base_test_mask].sum(axis=0).astype(np.float64)
    initial_train_error = base_train_counts - target_train
    initial_val_error = base_val_counts - target_val
    fixed_test_error = base_test_counts - target_test
    fixed_test_objective = _objective(fixed_test_error, weights)
    initial_objective = (
        _objective(initial_train_error, weights)
        + _objective(initial_val_error, weights)
        + fixed_test_objective
    )

    bucket_members: dict[tuple[int, ...], dict[str, list[int]]] = defaultdict(
        lambda: {"train": [], "val": []}
    )
    for group_index in np.flatnonzero(dev_mask & group_movable):
        split_name = str(base_splits[group_index])
        if split_name not in {"train", "val"}:
            continue
        signature = (int(group_sizes[group_index]),) + tuple(
            int(value) for value in type_counts[group_index]
        )
        bucket_members[signature][split_name].append(int(group_index))

    usable = [
        value
        for value in bucket_members.values()
        if value["train"] and value["val"]
    ]
    if not usable:
        raise RuntimeError("no group-preserving swap bucket is available")
    bucket_prob = np.asarray(
        [np.sqrt(len(item["train"]) * len(item["val"])) for item in usable],
        dtype=np.float64,
    )
    bucket_prob /= bucket_prob.sum()
    bucket_cdf = np.cumsum(bucket_prob)
    bucket_cdf[-1] = 1.0

    best_assignment = base_splits.copy()
    best_objective = initial_objective
    best_accepted = 0
    best_restart = 0

    for restart in range(restarts):
        rng = np.random.default_rng(seed + restart)
        assignment = base_splits.copy()
        pools = [
            {"train": item["train"].copy(), "val": item["val"].copy()}
            for item in usable
        ]
        train_counts = base_train_counts.copy()
        val_counts = base_val_counts.copy()
        train_error = initial_train_error.copy()
        val_error = initial_val_error.copy()
        objective = initial_objective
        accepted = 0

        for attempt in range(1, iterations + 1):
            bucket_index = int(
                np.searchsorted(bucket_cdf, rng.random(), side="right")
            )
            pool = pools[bucket_index]
            train_pos = int(rng.integers(len(pool["train"])))
            val_pos = int(rng.integers(len(pool["val"])))
            train_group = pool["train"][train_pos]
            val_group = pool["val"][val_pos]
            diff = (
                group_features[train_group].astype(np.float64)
                - group_features[val_group].astype(np.float64)
            )
            affected = np.flatnonzero(diff)
            if not len(affected):
                continue
            delta = float(
                np.sum(
                    weights[affected]
                    * (
                        2.0
                        * (val_error[affected] - train_error[affected])
                        * diff[affected]
                        + 2.0 * diff[affected] ** 2
                    )
                )
            )
            if delta < -1e-15:
                assignment[train_group] = "val"
                assignment[val_group] = "train"
                pool["train"][train_pos] = val_group
                pool["val"][val_pos] = train_group
                train_counts -= diff
                val_counts += diff
                train_error -= diff
                val_error += diff
                objective += delta
                accepted += 1
            if progress_every and attempt % progress_every == 0:
                print(
                    f"split restart={restart + 1}/{restarts} "
                    f"attempt={attempt}/{iterations} accepted={accepted} "
                    f"objective={objective:.8f}"
                )

        if objective < best_objective - 1e-12:
            best_assignment = assignment
            best_objective = objective
            best_accepted = accepted
            best_restart = restart

    if not np.array_equal(best_assignment == "test", base_splits == "test"):
        raise AssertionError("fixed test membership changed")
    nonmovable = ~group_movable
    if not np.array_equal(
        best_assignment[nonmovable], base_splits[nonmovable]
    ):
        raise AssertionError("a nonmovable group changed its inherited split")
    final_train_rows = int(group_sizes[best_assignment == "train"].sum())
    final_val_rows = int(group_sizes[best_assignment == "val"].sum())
    if final_train_rows != int(target_train_rows):
        raise AssertionError("training row count changed during optimization")
    if final_val_rows != int(target_val_rows):
        raise AssertionError("validation row count changed during optimization")
    if not group_holdout_eligible[best_assignment == "val"].all():
        raise AssertionError("an ineligible group moved into validation")

    # Recompute the final objective independently to catch pool/state drift.
    final_train_counts = group_features[best_assignment == "train"].sum(
        axis=0
    ).astype(np.float64)
    final_val_counts = group_features[best_assignment == "val"].sum(
        axis=0
    ).astype(np.float64)
    recomputed_objective = (
        _objective(final_train_counts - target_train, weights)
        + _objective(final_val_counts - target_val, weights)
        + fixed_test_objective
    )
    if not np.isclose(
        recomputed_objective,
        best_objective,
        rtol=1e-10,
        atol=1e-10,
    ):
        raise AssertionError(
            "incremental objective drift: "
            f"{best_objective} != {recomputed_objective}"
        )
    best_objective = float(recomputed_objective)

    # Same-size, same-type-vector swaps must preserve every per-type count.
    for split_name in ("train", "val", "test"):
        before = type_counts[base_splits == split_name].sum(axis=0)
        after = type_counts[best_assignment == split_name].sum(axis=0)
        if not np.array_equal(before, after):
            raise AssertionError(f"per-type count changed in {split_name}")

    return SplitOptimizationResult(
        group_ids=group_ids,
        group_assignment=best_assignment,
        feature_names=tuple(feature_names),
        initial_objective=float(initial_objective),
        final_objective=float(best_objective),
        accepted_swaps=int(best_accepted),
        attempted_swaps=int(iterations) * int(restarts),
        restart=int(best_restart),
    )


def apply_group_assignment(
    frame: pd.DataFrame,
    result: SplitOptimizationResult,
) -> pd.DataFrame:
    assignment = result.assignment_series()
    out = frame.copy()
    out["base_split"] = out["split"].astype(str)
    out["split"] = out["group_id"].astype(str).map(assignment)
    if out["split"].isna().any():
        raise AssertionError("group assignment join produced missing values")
    return out


def build_feature_balance_audit(
    frame: pd.DataFrame,
    row_features: np.ndarray,
    feature_names: Sequence[str],
    families: Sequence[str],
    *,
    swap_eligible_column: str = "eligible_for_training",
    holdout_eligibility_column: str = "eligible_for_training",
) -> pd.DataFrame:
    if len(frame) != len(row_features):
        raise AssertionError("audit row length mismatch")
    splits = ("train", "val", "test")
    group_codes, _ = pd.factorize(frame["group_id"].astype(str), sort=True)
    group_count = int(group_codes.max()) + 1
    group_sizes = np.bincount(group_codes, minlength=group_count).astype(np.int32)
    group_features = _sum_binary_features_by_group(
        group_codes, row_features, group_count, group_sizes
    )
    group_support = (group_features > 0).sum(axis=0)
    total = row_features.sum(axis=0).astype(np.int64)

    split_codes = pd.Categorical(
        frame["split"], categories=list(splits)
    ).codes
    if (split_codes < 0).any():
        raise AssertionError("unexpected split in balance audit")
    group_split_min = np.full(group_count, len(splits), dtype=np.int8)
    group_split_max = np.full(group_count, -1, dtype=np.int8)
    np.minimum.at(group_split_min, group_codes, split_codes)
    np.maximum.at(group_split_max, group_codes, split_codes)
    if not np.array_equal(group_split_min, group_split_max):
        raise AssertionError("group leaks across splits in balance audit")
    group_swap_eligible = _group_all_true(
        frame, swap_eligible_column, group_codes, group_sizes
    )
    group_holdout_eligible = _group_all_true(
        frame, holdout_eligibility_column, group_codes, group_sizes
    )
    if np.any(group_swap_eligible & ~group_holdout_eligible):
        raise AssertionError(
            "swap eligibility cannot override holdout/training ineligibility"
        )
    group_movable = group_swap_eligible & group_holdout_eligible
    split_row_counts = frame["split"].value_counts().reindex(splits, fill_value=0)
    ratio_target = {
        split_name: int(split_row_counts[split_name]) / len(frame)
        for split_name in splits
    }

    rows = []
    for feature_index, (name, family) in enumerate(zip(feature_names, families)):
        record = {
            "feature": name,
            "family": family,
            "total_rows": int(total[feature_index]),
            "independent_groups": int(group_support[feature_index]),
        }
        deviations = []
        for split_name in splits:
            mask = frame["split"].eq(split_name).to_numpy()
            count = int(row_features[mask, feature_index].sum())
            feature_share = count / max(1, int(total[feature_index]))
            share_deviation = feature_share - ratio_target[split_name]
            split_rows = int(split_row_counts[split_name])
            split_prevalence = count / max(1, split_rows)
            global_prevalence = int(total[feature_index]) / len(frame)
            record[f"{split_name}_rows"] = count
            record[f"{split_name}_share"] = feature_share
            record[f"{split_name}_share_error_pp"] = 100.0 * share_deviation
            record[f"{split_name}_prevalence"] = split_prevalence
            record[f"{split_name}_prevalence_error_pp"] = 100.0 * (
                split_prevalence - global_prevalence
            )
            deviations.append(abs(share_deviation))
        record["global_prevalence"] = int(total[feature_index]) / len(frame)
        record["max_abs_share_error_pp"] = 100.0 * max(deviations)
        record["max_abs_prevalence_error_pp"] = max(
            abs(float(record[f"{split_name}_prevalence_error_pp"]))
            for split_name in splits
        )
        record["three_way_coverage"] = all(
            int(record[f"{split_name}_rows"]) > 0 for split_name in splits
        )

        feature_groups = group_features[:, feature_index] > 0
        test_groups = int(
            np.sum(feature_groups & (group_split_min == splits.index("test")))
        )
        movable_dev_groups = int(
            np.sum(
                feature_groups
                & (group_split_min != splits.index("test"))
                & group_movable
            )
        )
        actual_train_only_groups = int(
            np.sum(
                feature_groups
                & (group_split_min != splits.index("test"))
                & ~group_holdout_eligible
            )
        )
        frozen_dev_groups = (
            feature_groups
            & (group_split_min != splits.index("test"))
            & ~group_movable
        )
        frozen_train_groups = int(
            np.sum(frozen_dev_groups & (group_split_min == splits.index("train")))
        )
        frozen_val_groups = int(
            np.sum(frozen_dev_groups & (group_split_min == splits.index("val")))
        )
        swap_ineligible_dev_groups = int(np.sum(frozen_dev_groups))
        frozen_dev_coverage = int(frozen_train_groups > 0) + int(
            frozen_val_groups > 0
        )
        if frozen_dev_coverage == 2:
            max_dev_coverage = 2
        elif frozen_dev_coverage == 1:
            max_dev_coverage = 1 + int(movable_dev_groups > 0)
        else:
            max_dev_coverage = min(2, movable_dev_groups)
        max_coverage = int(test_groups > 0) + max_dev_coverage
        record["fixed_test_groups"] = test_groups
        # Retain the old column names for artifact compatibility, but give
        # train_only its actual training-eligibility meaning.  New columns make
        # frozen eligible validation groups visible instead of misclassifying
        # them as train-only quarantine groups.
        record["eligible_development_groups"] = movable_dev_groups
        record["train_only_development_groups"] = actual_train_only_groups
        record["movable_development_groups"] = movable_dev_groups
        record["swap_ineligible_development_groups"] = (
            swap_ineligible_dev_groups
        )
        record["training_ineligible_development_groups"] = (
            actual_train_only_groups
        )
        record["frozen_train_development_groups"] = frozen_train_groups
        record["frozen_validation_development_groups"] = frozen_val_groups
        record["max_achievable_split_coverage"] = max_coverage
        record["max_achievable_split_coverage_upper_bound"] = max_coverage
        record["three_way_coverage_feasible"] = max_coverage >= 3
        rows.append(record)
    return pd.DataFrame(rows).sort_values(
        ["family", "total_rows", "feature"],
        ascending=[True, False, True],
        kind="mergesort",
    ).reset_index(drop=True)


def assert_split_hard_gates(
    frame: pd.DataFrame,
    *,
    expected_counts: dict[str, int],
    fixed_test_stems: Iterable[str],
) -> dict[str, bool]:
    required = {"stem", "group_id", "split", "eligible_for_training"}
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"hard-gate columns missing: {sorted(missing)}")
    expected_test = sorted(str(value) for value in fixed_test_stems)
    actual_test = sorted(frame.loc[frame["split"].eq("test"), "stem"].astype(str))
    actual_counts = (
        frame["split"]
        .value_counts()
        .reindex(expected_counts, fill_value=0)
        .astype(int)
        .to_dict()
    )
    split_values = set(frame["split"].dropna().astype(str))
    ineligible = ~frame["eligible_for_training"].astype(bool)
    sha_leakage_zero = True
    if "image_sha256" in frame.columns:
        sha_leakage_zero = (
            int(
                frame.groupby("image_sha256", observed=True)["split"]
                .nunique()
                .max()
            )
            == 1
        )
    scene_leakage_zero = True
    if "scene_group_id" in frame.columns:
        valid_scene = (
            frame["scene_group_id"].notna()
            & frame["scene_group_id"].astype(str).str.strip().ne("")
        )
        if valid_scene.any():
            scene_leakage_zero = (
                int(
                    frame.loc[valid_scene]
                    .groupby("scene_group_id", observed=True)["split"]
                    .nunique()
                    .max()
                )
                == 1
            )
    base_test_unchanged = True
    if "base_split" in frame.columns:
        base_test_unchanged = bool(
            np.array_equal(
                frame["split"].eq("test").to_numpy(),
                frame["base_split"].astype(str).eq("test").to_numpy(),
            )
        )

    assertions = {
        "row_count_exact": len(frame) == sum(int(v) for v in expected_counts.values()),
        "all_stems_present": bool(frame["stem"].notna().all()),
        "all_stems_unique": bool(frame["stem"].is_unique),
        "all_group_ids_present": bool(frame["group_id"].notna().all()),
        "split_values_exact": split_values == set(expected_counts),
        "split_counts_exact": actual_counts == {
            name: int(value) for name, value in expected_counts.items()
        },
        "group_leakage_zero": (
            int(
                frame.groupby("group_id", observed=True)["split"]
                .nunique()
                .max()
            )
            == 1
        ),
        "sha_leakage_zero": bool(sha_leakage_zero),
        "scene_leakage_zero": bool(scene_leakage_zero),
        "fixed_test_exact": actual_test == expected_test,
        "base_test_membership_unchanged": base_test_unchanged,
        "validation_eligible": bool(
            frame.loc[frame["split"].eq("val"), "eligible_for_training"].all()
        ),
        "test_eligible": bool(
            frame.loc[frame["split"].eq("test"), "eligible_for_training"].all()
        ),
        "ineligible_rows_train_only": bool(
            frame.loc[ineligible, "split"].eq("train").all()
        ),
    }
    if not all(assertions.values()):
        raise AssertionError(f"split hard gate failed: {assertions}")
    return assertions


def synthetic_self_test() -> dict[str, float | int | bool]:
    """Exercise leakage, multi-label handling, exact counts, and determinism."""
    rows = []
    types = TYPE_ORDER
    for index in range(120):
        type_name = types[index % len(types)]
        split = "test" if index % 10 == 0 else ("val" if index % 10 == 1 else "train")
        tags = [REAL_OBSTRUCTIONS[index % 3]]
        if index % 7 == 0:
            tags.append(REAL_OBSTRUCTIONS[(index + 1) % 3])
        rows.append(
            {
                "stem": f"{type_name}_{index:04d}",
                "type": type_name,
                "water_bin": index % 10,
                "obstruction": tags,
                "image_sha256": f"{index:064x}",
                "group_id": f"sha256:{index:064x}",
                "split": split,
                "eligible_for_training": index != 2,
            }
        )
    frame = pd.DataFrame(rows)
    # Same-type duplicate-image groups exercise indivisibility and size/type
    # buckets.  One frozen group is inherited by each split so swap eligibility
    # cannot be mistaken for holdout/training eligibility.
    for members, digest in (
        ([1, 41], f"{10_001:064x}"),
        ([9, 49], f"{10_002:064x}"),
        ([0, 40], f"{10_003:064x}"),
    ):
        frame.loc[members, "image_sha256"] = digest
        frame.loc[members, "group_id"] = "sha256:" + digest
    frame = add_normalized_obstruction_columns(frame)
    frozen_group_ids = {
        f"sha256:{10_001:064x}",  # inherited validation conflict
        f"sha256:{10_002:064x}",  # inherited training conflict
        f"sha256:{10_003:064x}",  # inherited fixed-test conflict
    }
    frame["_swap_eligible"] = (
        frame["eligible_for_training"].astype(bool)
        & ~frame["group_id"].isin(frozen_group_ids)
    )
    frozen_base_assignment = (
        frame.loc[frame["group_id"].isin(frozen_group_ids), ["group_id", "split"]]
        .drop_duplicates()
        .set_index("group_id")["split"]
        .sort_index()
    )
    if len(frozen_base_assignment) != 3:
        raise AssertionError("synthetic frozen group setup failed")
    if set(frozen_base_assignment) != {"train", "val", "test"}:
        raise AssertionError("synthetic frozen groups must cover all splits")
    matrix, names, families = build_row_feature_matrix(
        frame, min_pair_group_support=2
    )
    if not any(name.startswith("obs_pair=") for name in names):
        raise AssertionError("synthetic pair feature was not selected")
    result = optimize_fixed_test_group_swaps(
        frame,
        matrix,
        names,
        families,
        target_val_rows=12,
        iterations=5_000,
        restarts=2,
        seed=123,
        swap_eligible_column="_swap_eligible",
    )
    final = apply_group_assignment(frame, result)
    assertions = assert_split_hard_gates(
        final,
        expected_counts={"train": 96, "val": 12, "test": 12},
        fixed_test_stems=frame.loc[frame["split"].eq("test"), "stem"],
    )
    if result.final_objective > result.initial_objective + 1e-12:
        raise AssertionError("synthetic objective worsened")
    if final.groupby("group_id", observed=True)["split"].nunique().max() != 1:
        raise AssertionError("synthetic duplicate group leaked")
    if final.loc[2, "split"] != "train":
        raise AssertionError("synthetic ineligible row left train")
    frozen_final_assignment = (
        final.loc[final["group_id"].isin(frozen_group_ids), ["group_id", "split"]]
        .drop_duplicates()
        .set_index("group_id")["split"]
        .sort_index()
    )
    if not frozen_final_assignment.equals(frozen_base_assignment):
        raise AssertionError("synthetic nonmovable group changed split")
    audit = build_feature_balance_audit(
        final,
        matrix,
        names,
        families,
        swap_eligible_column="_swap_eligible",
    )
    if int(audit["frozen_validation_development_groups"].max()) < 1:
        raise AssertionError("synthetic frozen validation group was not audited")
    if int(audit["training_ineligible_development_groups"].max()) < 1:
        raise AssertionError("synthetic training-ineligible group was not audited")

    # Reordering the input rows must not change the sorted group assignment.
    shuffled = frame.sample(frac=1.0, random_state=987).reset_index(drop=True)
    shuffled_matrix, shuffled_names, shuffled_families = build_row_feature_matrix(
        shuffled, min_pair_group_support=2
    )
    second = optimize_fixed_test_group_swaps(
        shuffled,
        shuffled_matrix,
        shuffled_names,
        shuffled_families,
        target_val_rows=12,
        iterations=5_000,
        restarts=2,
        seed=123,
        swap_eligible_column="_swap_eligible",
    )
    if not np.array_equal(result.group_ids, second.group_ids):
        raise AssertionError("group order changed after input shuffle")
    if not np.array_equal(result.group_assignment, second.group_assignment):
        raise AssertionError("split optimizer is not input-order deterministic")
    return {
        "initial_objective": result.initial_objective,
        "final_objective": result.final_objective,
        "accepted_swaps": result.accepted_swaps,
        "hard_gates": all(assertions.values()),
        "deterministic": True,
        "nonmovable_groups_frozen": True,
        "frozen_validation_group_audited": True,
    }


if __name__ == "__main__":
    print(synthetic_self_test())


{'initial_objective': 26.21866020761246, 'final_objective': 21.374194232987314, 'accepted_swaps': 9, 'hard_gates': True, 'deterministic': True, 'nonmovable_groups_frozen': True, 'frozen_validation_group_audited': True}


In [ ]:
base_split = pd.read_parquet(BASE_SPLIT_PATH)
raw_obstruction = pq.read_table(
    RAW_PATH, columns=["stem", "obstruction"]
).to_pandas()
if not base_split["stem"].is_unique or not raw_obstruction["stem"].is_unique:
    raise AssertionError("stem은 양쪽 입력에서 유일해야 합니다")
split_df = base_split.merge(
    raw_obstruction, on="stem", how="left", validate="one_to_one", indicator=True
)
if not split_df["_merge"].eq("both").all():
    raise AssertionError("raw obstruction join 완전성 실패")
split_df = split_df.drop(columns="_merge")
if len(split_df) != EXPECTED_RAW_ROWS:
    raise AssertionError("전체 행 수 불일치")
split_df = add_normalized_obstruction_columns(
    split_df, source_column="obstruction", vocabulary=REAL_OBSTRUCTIONS
)
unknown_count = int(split_df["unknown_obstruction_labels"].map(bool).sum())
if unknown_count:
    raise AssertionError(f"알 수 없는 obstruction label: {unknown_count}행")
print("multi-obstacle rows:", int(split_df["multi_obstacle"].sum()))
print("contradiction rows:", int(split_df["contradiction"].sum()))
print("empty/unknown rows:", int(split_df["empty_or_unknown"].sum()))
split_df["_obstruction_signature"] = [
    json.dumps(
        {
            "labels": list(labels),
            "none_only": bool(none_only),
            "contradiction": bool(contradiction),
            "empty_or_unknown": bool(empty_or_unknown),
        },
        ensure_ascii=False, sort_keys=True, separators=(",", ":"),
    )
    for labels, none_only, contradiction, empty_or_unknown in zip(
        split_df["obstruction_labels"], split_df["none_only"],
        split_df["contradiction"], split_df["empty_or_unknown"],
    )
]
split_df["_obstruction_target_signature"] = split_df[
    "obstruction_targets"
].map(lambda values: json.dumps(list(values), separators=(",", ":")))
group_label_audit = split_df.groupby("group_id", observed=True).agg(
    type_nunique=("type", "nunique"),
    waterlevel_nunique=("waterlevel_m", "nunique"),
    obstruction_target_nunique=("_obstruction_target_signature", "nunique"),
    obstruction_signature_nunique=("_obstruction_signature", "nunique"),
)
type_conflict_groups = set(group_label_audit.index[
    group_label_audit["type_nunique"].gt(1)
].astype(str))
waterlevel_conflict_groups = set(group_label_audit.index[
    group_label_audit["waterlevel_nunique"].gt(1)
].astype(str))
obstruction_target_conflict_groups = set(group_label_audit.index[
    group_label_audit["obstruction_target_nunique"].gt(1)
].astype(str))
obstruction_provenance_conflict_groups = set(group_label_audit.index[
    group_label_audit["obstruction_signature_nunique"].gt(1)
    & group_label_audit["obstruction_target_nunique"].eq(1)
].astype(str))
conflict_groups = (
    type_conflict_groups
    | waterlevel_conflict_groups
    | obstruction_target_conflict_groups
    | obstruction_provenance_conflict_groups
)
group_ids = split_df["group_id"].astype(str)
split_df["group_type_conflict"] = group_ids.isin(type_conflict_groups)
split_df["group_waterlevel_target_conflict"] = group_ids.isin(
    waterlevel_conflict_groups
)
split_df["group_obstruction_target_conflict"] = group_ids.isin(
    obstruction_target_conflict_groups
)
split_df["group_obstruction_provenance_conflict"] = group_ids.isin(
    obstruction_provenance_conflict_groups
)
split_df["group_label_conflict"] = group_ids.isin(conflict_groups)
# Exact-image duplicate groups may inherit inconsistent annotations from v3.
# Preserve every row for water-level regression, freeze the entire group in
# its base split, and remove only contradictory obstacle targets from the
# auxiliary BCE contract.
split_df["obstruction_mask"] = [
    [0] * len(REAL_OBSTRUCTIONS) if conflict else list(mask)
    for mask, conflict in zip(
        split_df["obstruction_mask"],
        split_df["group_obstruction_target_conflict"],
    )
]
conflict_assignment_before = (
    split_df.loc[split_df["group_label_conflict"], ["group_id", "split"]]
    .drop_duplicates()
    .set_index("group_id")["split"]
    .astype(str)
    .sort_index()
)
split_df["_swap_eligible"] = (
    split_df["eligible_for_training"].astype(bool)
    & ~split_df["group_label_conflict"]
)
conflict_reason_group_counts = {
    "type": len(type_conflict_groups),
    "waterlevel_target": len(waterlevel_conflict_groups),
    "obstruction_target": len(obstruction_target_conflict_groups),
    "obstruction_provenance_only": len(obstruction_provenance_conflict_groups),
    "any": len(conflict_groups),
}
print("label-conflict groups by reason:", conflict_reason_group_counts)
display(
    split_df.loc[split_df["group_label_conflict"]]
    .groupby("split", observed=True)
    .agg(rows=("group_id", "size"), groups=("group_id", "nunique"))
    .reindex(["train", "val", "test"], fill_value=0)
)

row_features, feature_names, feature_families = build_row_feature_matrix(
    split_df, type_order=TYPE_ORDER, vocabulary=REAL_OBSTRUCTIONS,
    min_pair_group_support=MIN_PAIR_GROUP_SUPPORT,
)
fixed_test_stems = split_df.loc[
    split_df["split"].eq("test"), "stem"
].astype(str).tolist()
optimization = optimize_fixed_test_group_swaps(
    split_df,
    row_features,
    feature_names,
    feature_families,
    target_val_rows=EXPECTED_COUNTS["val"],
    iterations=SWAP_ITERATIONS,
    restarts=SWAP_RESTARTS,
    seed=SEED,
    progress_every=50_000,
    swap_eligible_column="_swap_eligible",
)
split_df = apply_group_assignment(split_df, optimization)
conflict_assignment_after = (
    split_df.loc[split_df["group_label_conflict"], ["group_id", "split"]]
    .drop_duplicates()
    .set_index("group_id")["split"]
    .astype(str)
    .sort_index()
)
if not conflict_assignment_after.equals(conflict_assignment_before):
    raise AssertionError("label-conflict group의 기존 split이 변경되었습니다")
split_assertions = assert_split_hard_gates(
    split_df,
    expected_counts=EXPECTED_COUNTS,
    fixed_test_stems=fixed_test_stems,
)
if optimization.final_objective > optimization.initial_objective + 1e-12:
    raise AssertionError("multilabel objective가 악화되었습니다")
if split_df.groupby(["type", "split"], observed=True).size().to_dict() != (
    base_split.groupby(["type", "split"], observed=True).size().to_dict()
):
    raise AssertionError("유형별 split 수량이 변경되었습니다")
if int(split_df.loc[split_df["split"].eq("train"), "eligible_for_training"].sum()) != EXPECTED_EFFECTIVE_COUNTS["train"]:
    raise AssertionError("effective train count 불일치")

balance_audit = build_feature_balance_audit(
    split_df, row_features, feature_names, feature_families,
    swap_eligible_column="_swap_eligible",
)
infeasible = balance_audit.loc[
    (~balance_audit["three_way_coverage_feasible"])
    | (
        balance_audit["three_way_coverage_feasible"]
        & ~balance_audit["three_way_coverage"]
    )
].copy()
print("objective:", optimization.initial_objective, "->", optimization.final_objective)
print("accepted swaps:", optimization.accepted_swaps)
display(split_df.groupby(["type", "split"], observed=True).size().unstack(fill_value=0))
display(balance_audit.sort_values("max_abs_share_error_pp", ascending=False).head(30))


multi-obstacle rows: 75954
contradiction rows: 574
empty/unknown rows: 10
label-conflict groups by reason: {'type': 0, 'waterlevel_target': 233, 'obstruction_target': 741, 'obstruction_provenance_only': 0, 'any': 929}


       rows  groups
split              
train  1450     725
val     216     108
test    192      96

split restart=1/3 attempt=50000/300000 accepted=2501 objective=8.08041025
split restart=1/3 attempt=100000/300000 accepted=2613 objective=8.07554209
split restart=1/3 attempt=150000/300000 accepted=2639 objective=8.07523206
split restart=1/3 attempt=200000/300000 accepted=2650 objective=8.07504436
split restart=1/3 attempt=250000/300000 accepted=2657 objective=8.07483549
split restart=1/3 attempt=300000/300000 accepted=2666 objective=8.07479265
split restart=2/3 attempt=50000/300000 accepted=2440 objective=8.43812448
split restart=2/3 attempt=100000/300000 accepted=2598 objective=8.07662944
split restart=2/3 attempt=150000/300000 accepted=2647 objective=8.07586807
split restart=2/3 attempt=200000/300000 accepted=2702 objective=8.07462638
split restart=2/3 attempt=250000/300000 accepted=2704 objective=8.07462629
split restart=2/3 attempt=300000/300000 accepted=2736 objective=8.07405789
split restart=3/3 attempt=100000/300000 accepted=2679 objective=8.07542455
split restart=3/3 attempt=2

split   test  train    val
type                      
A      10522  84176  10522
B       3727  29813   3726
C       9194  73553   9194
E       2351  18805   2350
F        878   7030    879
G       1098   8781   1098
H       1031   8249   1031
S       2008  16061   2008

             feature         family  total_rows  independent_groups  \
190    type_obs=F|야간  type_obstacle           1                   1   
197    type_obs=B|야간  type_obstacle           0                   0   
198    type_obs=F|기타  type_obstacle           0                   0   
199  type_obs=F|달빛반사  type_obstacle           0                   0   
196    type_obs=B|안개  type_obstacle           0                   0   
201  type_obs=F|표면손상  type_obstacle           0                   0   
202  type_obs=G|달빛반사  type_obstacle           0                   0   
203    type_obs=G|사람  type_obstacle           0                   0   
204    type_obs=G|탁류  type_obstacle           0                   0   
209    type_obs=S|안개  type_obstacle           0                   0   
210    type_obs=S|야간  type_obstacle           0                   0   
208    type_obs=S|사람  type_obstacle           0                   0   
207   type_obs=S|부착물  type_obstacle           0                   0   
206  t

## 3. v4 split·평가 metadata·감사표 봉인

test truth와 장애물은 private prefix에만 둡니다. train/validation 학습용 shard에는 보조
target을 넣지만 익명 test image seal은 기존 byte를 그대로 참조합니다.


In [ ]:
SPLIT_ALGORITHM_VERSION = "fixed-v3-test-group-type-exact-multilabel-swap-v2"
VOCABULARY_SHA256 = canonical_sha256(REAL_OBSTRUCTIONS)
SPLIT_CONFIG = {
    "schema_version": "dcc-m3-v4-split-config-v2",
    "run_id": RUN_ID,
    "input_version": INPUT_VERSION,
    "base_split_version": BASE_SPLIT_VERSION,
    "output_version": OUTPUT_VERSION,
    "test_policy": "fixed_v3_test",
    "ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
    "expected_counts": EXPECTED_COUNTS,
    "algorithm": SPLIT_ALGORITHM_VERSION,
    "seed": SEED,
    "swap_iterations": SWAP_ITERATIONS,
    "swap_restarts": SWAP_RESTARTS,
    "min_pair_group_support": MIN_PAIR_GROUP_SUPPORT,
    "serialization_versions": SERIALIZATION_VERSIONS,
    "label_conflict_policy": {
        "split_assignment": "freeze_inherited_base_split",
        "swap": "exclude_all_annotation_conflict_groups",
        "water_level_regression": "retain_all_rows",
        "obstacle_auxiliary": "mask_obstruction_target_conflict_rows",
    },
    "feature_names": list(feature_names),
    "feature_families": list(feature_families),
    "obstruction_vocabulary": REAL_OBSTRUCTIONS,
    "obstruction_vocabulary_sha256": VOCABULARY_SHA256,
    "parent_raw_sha256": raw_descriptor["sha256"],
    "parent_split_sha256": split_descriptor["sha256"],
}
SPLIT_CONFIG_SHA256 = canonical_sha256(SPLIT_CONFIG)

private_split = split_df.drop(columns=[
    "unknown_obstruction_labels", "_obstruction_signature",
    "_obstruction_target_signature", "_swap_eligible",
]).sort_values("stem", kind="mergesort").reset_index(drop=True)
train_private = private_split.loc[private_split["split"].eq("train")].copy()
val_private = private_split.loc[private_split["split"].eq("val")].copy()
test_truth_metadata = private_split.loc[
    private_split["split"].eq("test"),
    [
        "stem", "type", "waterlevel_m", "group_id", "obstruction_labels",
        "none_only", "multi_obstacle", "contradiction", "empty_or_unknown",
        "group_obstruction_target_conflict",
    ],
].copy()
test_key_map = pd.read_parquet(BASE_TEST_KEY_MAP_PATH, columns=["sample_id", "stem"])
if len(test_key_map) != EXPECTED_COUNTS["test"] or not test_key_map["sample_id"].is_unique:
    raise AssertionError("base test key map count/unique 오류")
if canonical_sha256(sorted(test_key_map["sample_id"].astype(str))) != base_inference_seal["test_sample_id_sha256"]:
    raise AssertionError("base test sample ID SHA 불일치")
test_eval_metadata = test_key_map.merge(
    test_truth_metadata, on="stem", how="outer", validate="one_to_one", indicator=True
)
if len(test_eval_metadata) != EXPECTED_COUNTS["test"] or not test_eval_metadata["_merge"].eq("both").all():
    raise AssertionError("private test evaluation metadata join 완전성 실패")
test_eval_metadata = test_eval_metadata.drop(columns="_merge")
test_eval_metadata["obstruction"] = test_eval_metadata.pop("obstruction_labels")
PRIVATE_EVAL_COLUMNS = [
    "sample_id", "stem", "type", "waterlevel_m", "obstruction", "none_only",
    "multi_obstacle", "contradiction", "empty_or_unknown",
    "group_obstruction_target_conflict", "group_id",
]
test_eval_metadata = test_eval_metadata[PRIVATE_EVAL_COLUMNS]
if list(test_eval_metadata.columns) != PRIVATE_EVAL_COLUMNS:
    raise AssertionError("private test evaluation metadata schema 불일치")
if not test_eval_metadata["sample_id"].str[0].eq(test_eval_metadata["type"]).all():
    raise AssertionError("sample_id type 문자 불일치")

split_paths = {
    "splits": LOCAL_OUTPUT / f"splits_{OUTPUT_VERSION}.parquet",
    "train": LOCAL_OUTPUT / f"train_{OUTPUT_VERSION}.parquet",
    "val": LOCAL_OUTPUT / f"val_{OUTPUT_VERSION}.parquet",
    "test_eval_metadata": LOCAL_OUTPUT / f"test_eval_metadata_{OUTPUT_VERSION}.parquet",
    "balance_audit": LOCAL_OUTPUT / f"split_balance_audit_{OUTPUT_VERSION}.csv",
    "infeasible": LOCAL_OUTPUT / f"infeasible_strata_{OUTPUT_VERSION}.csv",
}
private_split.to_parquet(split_paths["splits"], index=False, compression="zstd")
train_private.to_parquet(split_paths["train"], index=False, compression="zstd")
val_private.to_parquet(split_paths["val"], index=False, compression="zstd")
test_eval_metadata.to_parquet(
    split_paths["test_eval_metadata"], index=False, compression="zstd"
)
balance_audit.to_csv(split_paths["balance_audit"], index=False, encoding="utf-8")
infeasible.to_csv(split_paths["infeasible"], index=False, encoding="utf-8")

split_artifact_specs = {
    "splits": ("private_v4_multilabel_splits", len(private_split)),
    "train": ("private_v4_multilabel_train", len(train_private)),
    "val": ("private_v4_multilabel_val", len(val_private)),
    "test_eval_metadata": ("m3_v4_private_test_eval_metadata", len(test_eval_metadata)),
    "balance_audit": ("private_v4_split_balance_audit", len(balance_audit)),
    "infeasible": ("private_v4_infeasible_strata", len(infeasible)),
}
split_artifacts = {}
for name, (artifact_type, row_count) in split_artifact_specs.items():
    object_name = f"{SPLIT_PREFIX}/{split_paths[name].name}"
    descriptor = upload_create_only(
        BUCKET, split_paths[name], object_name, artifact_type
    )
    descriptor["rows"] = int(row_count)
    split_artifacts[name] = descriptor

private_eval_contract_payload = {
    "schema_version": "dcc-m3-v4-private-test-eval-contract-v2",
    "claim_scope": "internal_fixed_holdout_after_prior_truth_exposure",
    "row_count": EXPECTED_COUNTS["test"],
    "columns": PRIVATE_EVAL_COLUMNS,
    "test_sample_id_sha256": base_inference_seal["test_sample_id_sha256"],
    "obstacle_vocabulary": REAL_OBSTRUCTIONS,
    "obstacle_vocabulary_sha256": VOCABULARY_SHA256,
    "test_eval_metadata": split_artifacts["test_eval_metadata"],
}
private_eval_contract_descriptor = upload_json_create_only(
    BUCKET,
    private_eval_contract_payload,
    f"{SPLIT_PREFIX}/test_eval_contract_{OUTPUT_VERSION}.json",
    "m3_v4_private_test_eval_contract",
)
split_artifacts["test_eval_contract"] = private_eval_contract_descriptor
BASE_TEST_KEY_MAP_PATH.unlink(missing_ok=True)

split_assignment_sha256 = canonical_sha256([
    {"stem": str(row.stem), "group_id": str(row.group_id), "split": str(row.split)}
    for row in private_split[["stem", "group_id", "split"]].itertuples(index=False)
])
conflict_assignment_sha256 = canonical_sha256([
    {"group_id": str(group_id), "base_split": str(split_name)}
    for group_id, split_name in conflict_assignment_before.items()
])
conflict_by_split = {
    split_name: {
        "groups": int(private_split.loc[
            private_split["group_label_conflict"]
            & private_split["split"].eq(split_name), "group_id"
        ].nunique()),
        "rows": int((
            private_split["group_label_conflict"]
            & private_split["split"].eq(split_name)
        ).sum()),
    }
    for split_name in ["train", "val", "test"]
}
split_seal_payload = {
    "status": "MULTILABEL_SPLIT_FROZEN",
    "schema_version": "dcc-m3-v4-multilabel-split-seal-v2",
    "output_version": OUTPUT_VERSION,
    "run_id": RUN_ID,
    "claim_scope": "fixed previously accessed v3 private test; model selection uses validation only",
    "config": SPLIT_CONFIG,
    "config_sha256": SPLIT_CONFIG_SHA256,
    "split_assignment_sha256": split_assignment_sha256,
    "counts": EXPECTED_COUNTS,
    "effective_counts": EXPECTED_EFFECTIVE_COUNTS,
    "type_split_counts": {
        type_name: {
            split_name: int((
                private_split["type"].eq(type_name)
                & private_split["split"].eq(split_name)
            ).sum())
            for split_name in ["train", "val", "test"]
        }
        for type_name in TYPE_ORDER
    },
    "obstruction_schema": {
        "labels": REAL_OBSTRUCTIONS,
        "encoding": "multi_hot_15_with_known_mask",
        "vocabulary_sha256": VOCABULARY_SHA256,
    },
    "optimization": {
        "initial_objective": optimization.initial_objective,
        "final_objective": optimization.final_objective,
        "accepted_swaps": optimization.accepted_swaps,
        "attempted_swaps": optimization.attempted_swaps,
        "best_restart": optimization.restart,
    },
    "label_conflicts": {
        "policy": SPLIT_CONFIG["label_conflict_policy"],
        "reason_group_counts": {
            name: int(value)
            for name, value in conflict_reason_group_counts.items()
        },
        "by_split": conflict_by_split,
        "base_assignment_sha256": conflict_assignment_sha256,
    },
    "fixed_test_stem_sha256": canonical_sha256(sorted(fixed_test_stems)),
    "train_group_sha256": canonical_sha256(sorted(
        private_split.loc[private_split["split"].eq("train"), "group_id"].astype(str).unique()
    )),
    "validation_group_sha256": canonical_sha256(sorted(
        private_split.loc[private_split["split"].eq("val"), "group_id"].astype(str).unique()
    )),
    "test_group_sha256": canonical_sha256(sorted(
        private_split.loc[private_split["split"].eq("test"), "group_id"].astype(str).unique()
    )),
    "parents": {
        "manifest_seal": MANIFEST_SEAL_DESCRIPTOR,
        "base_split_seal": BASE_SPLIT_SEAL_DESCRIPTOR,
        "base_dataset_seal": BASE_DATASET_SEAL_DESCRIPTOR,
        "base_inference_seal": BASE_INFERENCE_SEAL_DESCRIPTOR,
    },
    "artifacts": split_artifacts,
    "assertions": {
        **split_assertions,
        "objective_not_worse": optimization.final_objective <= optimization.initial_objective + 1e-12,
        "type_split_counts_unchanged": True,
        "effective_counts_exact": {
            "train": int(train_private["eligible_for_training"].sum()),
            "val": int(val_private["eligible_for_training"].sum()),
        } == EXPECTED_EFFECTIVE_COUNTS,
        "group_label_conflicts_frozen_to_base_split": bool(
            private_split.loc[
                private_split["group_label_conflict"], "split"
            ].astype(str).eq(private_split.loc[
                private_split["group_label_conflict"], "base_split"
            ].astype(str)).all()
        ),
        "obstruction_target_conflicts_masked": bool(
            private_split.loc[
                private_split["group_obstruction_target_conflict"],
                "obstruction_mask",
            ].map(lambda values: sum(int(value) for value in values) == 0).all()
        ),
        "test_eval_metadata_private": "/eval_private/" in split_artifacts["test_eval_metadata"]["uri"],
        "obstruction_not_from_error_analysis": True,
    },
}
if not all(split_seal_payload["assertions"].values()):
    raise AssertionError(f"v4 split seal assertion 실패: {split_seal_payload['assertions']}")
split_seal_descriptor = upload_json_create_only(
    BUCKET,
    split_seal_payload,
    f"{SPLIT_PREFIX}/split_seal_{OUTPUT_VERSION}.json",
    "m3_v4_multilabel_split_seal",
)
print("MULTILABEL SPLIT FROZEN")
print("split seal:", split_seal_descriptor["uri"])


MULTILABEL SPLIT FROZEN
split seal: gs://YOUR_GCS_BUCKET/dcc/eval_private/mission3/v4/v4_ml_fixedtest_v2/split_seal_v4_ml_fixedtest_v2.json


## 4. 기존 full train/validation shard를 v4 shard로 재포장

base shard를 한 개씩 내려받고 image byte와 기존 bbox metadata를 검증합니다. 새 split과
장애물 auxiliary target을 JSON에 기록한 뒤 create-only로 게시하고 로컬 입력 tar는 즉시
삭제합니다. 기존 test shard는 읽거나 다시 만들지 않습니다.


In [ ]:
DATASET_BUILD_CONFIG = {
    "schema_version": "dcc-m3-dataset-build-v2",
    "dataset_version": DATASET_VERSION,
    "data_profile": "full",
    "split_version": OUTPUT_VERSION,
    "split_seal_sha256": split_seal_descriptor["sha256"],
    "base_dataset_version": BASE_DATASET_VERSION,
    "base_dataset_seal_sha256": BASE_DATASET_SEAL_DESCRIPTOR["sha256"],
    "test_policy": "reuse_base_anonymous_test_bytes",
    "obstruction_schema": split_seal_payload["obstruction_schema"],
    "serialization_versions": SERIALIZATION_VERSIONS,
}
DATASET_CONFIG_SHA256 = canonical_sha256(DATASET_BUILD_CONFIG)
eligible_dev = private_split.loc[
    private_split["split"].isin(["train", "val"])
    & private_split["eligible_for_training"].astype(bool)
].copy()
if len(eligible_dev) != sum(EXPECTED_EFFECTIVE_COUNTS.values()):
    raise AssertionError("eligible development count 불일치")
lookup_columns = [
    "stem", "type", "waterlevel_m", "split", "group_id", "image_sha256",
    "width", "height", "detector_trainable", "obstruction_labels",
    "obstruction_targets", "obstruction_mask", "none_only", "multi_obstacle",
    "contradiction", "empty_or_unknown", "group_label_conflict",
    "group_type_conflict", "group_waterlevel_target_conflict",
    "group_obstruction_target_conflict",
    "group_obstruction_provenance_conflict",
]
row_by_stem = eligible_dev.set_index("stem")[lookup_columns[1:]].to_dict("index")

def verify_remote_descriptor(descriptor, expected_artifact_type=None):
    bucket_name, object_name = parse_gs_uri(descriptor["uri"])
    if bucket_name != BUCKET_NAME:
        raise AssertionError("remote descriptor bucket 불일치")
    blob = BUCKET.blob(object_name, generation=int(descriptor["generation"]))
    blob.reload()
    if int(blob.size) != descriptor_size(descriptor):
        raise AssertionError("remote descriptor byte 불일치")
    metadata = blob.metadata or {}
    if metadata.get("sha256") != descriptor["sha256"]:
        raise AssertionError("remote descriptor SHA metadata 불일치")
    if expected_artifact_type and metadata.get("artifact_type") != expected_artifact_type:
        raise AssertionError("remote descriptor artifact type 불일치")

def add_tar_bytes(handle, name, payload):
    info = tarfile.TarInfo(name=name)
    info.size = len(payload)
    info.mtime = 0
    info.mode = 0o644
    handle.addfile(info, io.BytesIO(payload))

def input_commit_object(input_descriptor):
    return f"{DATASET_PREFIX}/commits/{input_descriptor['sha256']}.json"

def verified_existing_repack_commit(input_descriptor):
    object_name = input_commit_object(input_descriptor)
    blob = BUCKET.blob(object_name)
    if not blob.exists():
        return None
    payload = json.loads(blob.download_as_bytes().decode("utf-8"))
    if payload.get("dataset_config_sha256") != DATASET_CONFIG_SHA256:
        raise RuntimeError("기존 repack commit과 현재 설정이 충돌합니다")
    if payload.get("input_sha256") != input_descriptor["sha256"]:
        raise AssertionError("repack commit input SHA 불일치")
    for descriptor in payload["outputs"]:
        verify_remote_descriptor(descriptor, "m3_v4_training_image_shard")
    return payload

base_train_val_descriptors = sorted(
    base_dataset_seal["shards"]["train"] + base_dataset_seal["shards"]["val"],
    key=lambda item: item["uri"],
)
if sum(int(item["samples"]) for item in base_train_val_descriptors) != len(eligible_dev):
    raise AssertionError("base shard와 v4 eligible development 수량 불일치")

repack_commits = []
for input_index, input_descriptor in enumerate(base_train_val_descriptors):
    existing = verified_existing_repack_commit(input_descriptor)
    if existing is not None:
        repack_commits.append(existing)
        if (input_index + 1) % 10 == 0:
            print("verified repack", input_index + 1, "/", len(base_train_val_descriptors))
        continue

    local_input_tar = LOCAL_STAGE / f"input-{input_index:05d}.tar"
    download_verified_descriptor(input_descriptor, local_input_tar)
    output_paths = {
        split_name: LOCAL_STAGE / f"output-{input_index:05d}-{split_name}.tar"
        for split_name in ["train", "val"]
    }
    output_handles = {
        split_name: tarfile.open(path, "w")
        for split_name, path in output_paths.items()
    }
    output_counts = {"train": 0, "val": 0}
    observed_keys = []
    try:
        with tarfile.open(local_input_tar, "r:") as source:
            members = [member for member in source.getmembers() if member.isfile()]
            names = {member.name for member in members}
            if len(names) != len(members):
                raise AssertionError("base tar member 이름 중복")
            json_names = sorted(name for name in names if name.endswith(".json"))
            if len(json_names) * 2 != len(names):
                raise AssertionError("base tar jpg/json pair 수 불일치")
            for json_name in json_names:
                key = Path(json_name).stem
                image_name = f"{key}.jpg"
                if image_name not in names or key not in row_by_stem:
                    raise AssertionError(f"base tar key 계약 오류: {key}")
                old_metadata = json.loads(
                    source.extractfile(json_name).read().decode("utf-8")
                )
                image_bytes = source.extractfile(image_name).read()
                row = row_by_stem[key]
                if str(old_metadata.get("stem")) != key:
                    raise AssertionError("base metadata stem 불일치")
                if str(old_metadata.get("type")) != str(row["type"]):
                    raise AssertionError("base metadata type 불일치")
                if not np.isclose(
                    np.float32(old_metadata["target_m"]),
                    np.float32(row["waterlevel_m"]),
                    rtol=0.0, atol=0.0,
                ):
                    raise AssertionError("base metadata target 불일치")
                if hashlib.sha256(image_bytes).hexdigest() != str(row["image_sha256"]):
                    raise AssertionError("base shard image SHA 불일치")
                split_name = str(row["split"])
                metadata = {
                    "stem": key,
                    "type": str(row["type"]),
                    "target_m": float(row["waterlevel_m"]),
                    "split": split_name,
                    "group_id": str(row["group_id"]),
                    "image_sha256": str(row["image_sha256"]),
                    "width": int(row["width"]),
                    "height": int(row["height"]),
                    "detector_trainable": bool(row["detector_trainable"]),
                    "mokja_boxes_xyxy_norm": old_metadata.get("mokja_boxes_xyxy_norm", []),
                    "obstruction_labels": list(row["obstruction_labels"]),
                    "obstruction_targets": [int(value) for value in row["obstruction_targets"]],
                    "obstruction_mask": [int(value) for value in row["obstruction_mask"]],
                    "none_only": bool(row["none_only"]),
                    "multi_obstacle": bool(row["multi_obstacle"]),
                    "contradiction": bool(row["contradiction"]),
                    "empty_or_unknown": bool(row["empty_or_unknown"]),
                    "group_label_conflict": bool(row["group_label_conflict"]),
                    "group_type_conflict": bool(row["group_type_conflict"]),
                    "group_waterlevel_target_conflict": bool(
                        row["group_waterlevel_target_conflict"]
                    ),
                    "group_obstruction_target_conflict": bool(
                        row["group_obstruction_target_conflict"]
                    ),
                    "group_obstruction_provenance_conflict": bool(
                        row["group_obstruction_provenance_conflict"]
                    ),
                }
                metadata_bytes = json.dumps(
                    metadata, ensure_ascii=False, sort_keys=True,
                    separators=(",", ":"), allow_nan=False,
                ).encode("utf-8")
                add_tar_bytes(output_handles[split_name], image_name, image_bytes)
                add_tar_bytes(output_handles[split_name], json_name, metadata_bytes)
                output_counts[split_name] += 1
                observed_keys.append(key)
    finally:
        for handle in output_handles.values():
            handle.close()
        local_input_tar.unlink(missing_ok=True)

    if len(observed_keys) != int(input_descriptor["samples"]):
        raise AssertionError("input shard sample count 불일치")
    output_descriptors = []
    for split_name, count in output_counts.items():
        path = output_paths[split_name]
        if count == 0:
            path.unlink(missing_ok=True)
            continue
        object_name = (
            f"{DATASET_PREFIX}/{split_name}/"
            f"{input_index:05d}-{input_descriptor['sha256'][:16]}.tar"
        )
        descriptor = upload_create_only(
            BUCKET, path, object_name, "m3_v4_training_image_shard"
        )
        descriptor.update({"split": split_name, "samples": int(count)})
        output_descriptors.append(descriptor)
        path.unlink(missing_ok=True)
    commit = {
        "status": "COMMITTED",
        "schema_version": "dcc-m3-v4-repack-commit-v1",
        "dataset_version": DATASET_VERSION,
        "dataset_config_sha256": DATASET_CONFIG_SHA256,
        "input_uri": input_descriptor["uri"],
        "input_sha256": input_descriptor["sha256"],
        "input_samples": int(input_descriptor["samples"]),
        "input_key_sha256": canonical_sha256(sorted(observed_keys)),
        "output_counts": output_counts,
        "outputs": output_descriptors,
    }
    upload_json_create_only(
        BUCKET, commit, input_commit_object(input_descriptor), "m3_v4_repack_commit"
    )
    repack_commits.append(commit)
    if (input_index + 1) % 10 == 0 or input_index + 1 == len(base_train_val_descriptors):
        print("repacked", input_index + 1, "/", len(base_train_val_descriptors))
    gc.collect()


repacked 10 / 177
repacked 20 / 177
repacked 30 / 177
repacked 40 / 177
repacked 50 / 177
repacked 60 / 177
repacked 70 / 177
repacked 80 / 177
repacked 90 / 177
repacked 100 / 177
repacked 110 / 177
repacked 120 / 177
repacked 130 / 177
repacked 140 / 177
repacked 150 / 177
repacked 160 / 177
repacked 170 / 177
repacked 177 / 177


## 5. v4 학습 dataset seal

shard count와 validation ID·target·obstacle target hash를 다시 계산합니다. test image는 기존
익명 inference seal descriptor만 참조하며 장애물과 정답을 넣지 않습니다.


In [ ]:
all_outputs = [descriptor for commit in repack_commits for descriptor in commit["outputs"]]
split_shards = {
    split_name: sorted(
        [item for item in all_outputs if item["split"] == split_name],
        key=lambda item: item["uri"],
    )
    for split_name in ["train", "val"]
}
actual_effective_counts = {
    split_name: sum(int(item["samples"]) for item in descriptors)
    for split_name, descriptors in split_shards.items()
}
if actual_effective_counts != EXPECTED_EFFECTIVE_COUNTS:
    raise AssertionError(f"v4 shard count 불일치: {actual_effective_counts}")

effective_train = eligible_dev.loc[eligible_dev["split"].eq("train")]
effective_val = eligible_dev.loc[eligible_dev["split"].eq("val")]
train_targets = np.asarray(effective_train["obstruction_targets"].tolist(), dtype=np.int64)
train_masks = np.asarray(effective_train["obstruction_mask"].tolist(), dtype=np.int64)
val_targets = np.asarray(effective_val["obstruction_targets"].tolist(), dtype=np.int64)
val_masks = np.asarray(effective_val["obstruction_mask"].tolist(), dtype=np.int64)
val_obstruction_records = [
    {
        "stem": str(row.stem),
        "obstruction_targets": [int(value) for value in row.obstruction_targets],
        "obstruction_mask": [int(value) for value in row.obstruction_mask],
    }
    for row in effective_val.sort_values("stem", kind="mergesort")[
        ["stem", "obstruction_targets", "obstruction_mask"]
    ].itertuples(index=False)
]
dataset_assertions = {
    "split_seal_frozen": split_seal_payload["status"] == "MULTILABEL_SPLIT_FROZEN",
    "source_input_shards_complete": sum(
        int(item["samples"]) for item in base_train_val_descriptors
    ) == len(eligible_dev),
    "repack_commits_complete": len(repack_commits) == len(base_train_val_descriptors),
    "train_count_exact": actual_effective_counts["train"] == EXPECTED_EFFECTIVE_COUNTS["train"],
    "validation_count_exact": actual_effective_counts["val"] == EXPECTED_EFFECTIVE_COUNTS["val"],
    "train_all_obstacles_have_known_positive": bool(
        ((train_targets * train_masks).sum(axis=0) > 0).all()
    ),
    "validation_all_obstacles_have_known_positive": bool(
        ((val_targets * val_masks).sum(axis=0) > 0).all()
    ),
    "anonymous_test_reused_without_read": True,
    "test_metadata_contract_sample_id_only": base_inference_seal.get("allowed_metadata") == ["sample_id"],
    "obstruction_absent_from_test_metadata": "obstruction" in base_inference_seal.get("forbidden_metadata", []),
}
if not all(dataset_assertions.values()):
    raise AssertionError(f"v4 dataset assertion 실패: {dataset_assertions}")

dataset_seal = {
    "status": "DATASET_COMPLETE",
    "schema_version": "dcc-m3-dataset-seal-v2",
    "dataset_version": DATASET_VERSION,
    "data_profile": "full",
    "config": DATASET_BUILD_CONFIG,
    "config_sha256": DATASET_CONFIG_SHA256,
    "split_seal": split_seal_descriptor,
    "upstream": {
        "split_seal_sha256": split_seal_descriptor["sha256"],
        "split_seal": split_seal_descriptor,
        "base_dataset_seal": BASE_DATASET_SEAL_DESCRIPTOR,
        "base_inference_seal": BASE_INFERENCE_SEAL_DESCRIPTOR,
    },
    "split_assignment_sha256": split_assignment_sha256,
    "counts": actual_effective_counts,
    "raw_counts": EXPECTED_COUNTS,
    "train_stem_sha256": canonical_sha256(sorted(effective_train["stem"].astype(str))),
    "train_group_sha256": split_seal_payload["train_group_sha256"],
    "validation_stem_sha256": canonical_sha256(sorted(effective_val["stem"].astype(str))),
    "validation_group_sha256": split_seal_payload["validation_group_sha256"],
    "validation_target_sha256": canonical_target_sha256(
        effective_val.rename(columns={"waterlevel_m": "target_m"}), "target_m"
    ),
    "validation_obstruction_sha256": canonical_sha256(val_obstruction_records),
    "obstruction_schema": split_seal_payload["obstruction_schema"],
    "serialization_versions": SERIALIZATION_VERSIONS,
    "train_obstruction_positive_counts": (train_targets * train_masks).sum(axis=0).astype(int).tolist(),
    "train_known_counts": train_masks.sum(axis=0).astype(int).tolist(),
    "shards": split_shards,
    "private": {
        "test_eval_metadata": split_artifacts["test_eval_metadata"],
        "test_eval_contract": private_eval_contract_descriptor,
    },
    "test_contract": {
        "inference_image_seal_sha256": BASE_INFERENCE_SEAL_DESCRIPTOR["sha256"],
        "test_sample_id_sha256": base_inference_seal["test_sample_id_sha256"],
        "row_count": int(base_inference_seal["counts"]['test']),
    },
    "inference": {
        "test_image_seal": BASE_INFERENCE_SEAL_DESCRIPTOR,
        "test_sample_id_sha256": base_inference_seal["test_sample_id_sha256"],
        "test_count": int(base_inference_seal["counts"]["test"]),
    },
    "assertions": dataset_assertions,
}
dataset_seal_descriptor = upload_json_create_only(
    BUCKET,
    dataset_seal,
    f"{DATASET_PREFIX}/dataset_seal.json",
    "m3_v4_dataset_seal",
)
print("V4 DATASET COMPLETE")
print("dataset seal:", dataset_seal_descriptor["uri"])
print("train/val shards:", {name: len(items) for name, items in split_shards.items()})
print("next: 04B GPU notebook")


V4 DATASET COMPLETE
dataset seal: gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v4-ml-fixedtest-v2/dataset_seal.json
train/val shards: {'train': 177, 'val': 175}
next: 04B GPU notebook
